# Problem Understanding

This notebook defines the real-world problem, project objective, system design, and academic relevance of the proposed end-to-end AI system for fraud detection and complaint analysis.

## 1. Problem Statement

Financial fraud is a serious issue in digital payment systems. Fraudulent transactions can cause direct monetary loss, reduce customer trust, and increase the operational burden on financial institutions. As transaction volume becomes larger, manual fraud detection becomes slower, less scalable, and more error-prone.

A second challenge is complaint handling. Customers often report suspicious activity, unauthorized payments, or service concerns through free-text complaints. These complaints are unstructured, which makes them difficult to process quickly and consistently using only manual review.

In a practical setting, these two problems are related. A suspicious transaction may need immediate action, while complaint text may provide supporting context about urgency, user concern, or unauthorized behavior. For this reason, the project aims to build a unified AI system that can analyze both structured transaction data and unstructured complaint text.

## 2. Project Objective

The main objective of this project is to develop an end-to-end AI system that can:

- predict the probability that a transaction is fraudulent
- convert the prediction into a business decision: `BLOCK`, `REVIEW`, or `APPROVE`
- analyze complaint text using NLP techniques
- provide a clear reason for the decision
- expose the complete workflow through a FastAPI service deployed on a VPS

This means the project is not limited to model training. It focuses on building a usable decision-support system that combines machine learning, NLP, business logic, and deployment.

## 3. ML Problem Formulation

- Problem Type: Binary classification
- Target Variable: `Class`
- Class Meaning:
  - `0` = Non-Fraud
  - `1` = Fraud
- Core Challenge: Severe class imbalance, with fraud cases representing a very small minority of transactions
- Learning Objective: Estimate the probability that a transaction is fraudulent so the system can support downstream actions such as `APPROVE`, `REVIEW`, or `BLOCK`

## 4. Evaluation Metric

Due to the severe class imbalance in fraud detection, accuracy is not an appropriate primary evaluation metric. A model can achieve very high accuracy by predicting most transactions as legitimate while still failing to identify fraudulent behavior.

- Primary Metric: `F1-score`
- Secondary Metric: `ROC-AUC`
- Business Focus: `Recall`, because catching fraud is more important than maximizing raw accuracy

Reason:

Missing fraud is costly, so the project should prioritize recall while still monitoring precision through the F1-score. `ROC-AUC` remains useful as a secondary ranking metric, but operational performance should emphasize the model's ability to detect fraudulent transactions.

## 5. Decision Logic

The fraud model is intended to support operational actions rather than only produce a score. For that reason, the predicted fraud probability will be converted into business decisions using threshold-based logic:

- `BLOCK`: probability > `0.85`
- `REVIEW`: probability from `0.40` to `0.85`
- `APPROVE`: probability < `0.40`

These thresholds are designed to balance two competing goals:

- fraud detection performance, especially recall
- false positives, which increase customer friction and manual review cost

This threshold design should be treated as an initial business rule. It can be refined later using validation results, precision-recall trade-offs, and operational tolerance for missed fraud versus unnecessary interventions.

## 6. System Overview

The proposed system contains four connected components:

### 6.1 Fraud Detection Module
This module uses machine learning to analyze transaction features and estimate the probability of fraud.

### 6.2 Decision Module
This module converts the fraud probability into an actionable decision:

- `BLOCK` for high-risk transactions
- `REVIEW` for uncertain or medium-risk transactions
- `APPROVE` for low-risk transactions

### 6.3 Complaint Analysis Module
This module uses NLP to process complaint text. It performs:

- sentiment analysis
- short complaint summarization

### 6.4 API Deployment Module
This module exposes the complete system through FastAPI so that transaction data and complaint text can be submitted and processed in a structured way.

## 7. Input and Output Definition

### 7.1 Input

The system receives two types of input.

**A. Transaction Data**

Structured transaction data is used by the fraud detection model. In this project, the transaction dataset contains numerical features such as:

- `Time`
- `Amount`
- anonymized variables such as `V1` to `V28`

For academic traceability, the transaction dataset used in this project is the Credit Card Fraud Detection dataset published on Kaggle by the ULB Machine Learning Group (2013). It contains 284,807 transactions made by European cardholders over two days in September 2013, and the target column is `Class`, where `1` represents fraud and `0` represents a legitimate transaction.

**B. Complaint Text**

Unstructured complaint text is used by the NLP module. Example complaint content may include:

- unauthorized transaction reports
- suspicious card activity
- customer concern or dissatisfaction

### 7.2 Output

The expected output of the system is:

```json
{
  "fraud_probability": float,
  "decision": "BLOCK/REVIEW/APPROVE",
  "reason": "explanation of decision",
  "complaint_analysis": {
    "sentiment": "positive/negative/neutral",
    "summary": "short summary"
  }
}
```

### 7.3 Output Interpretation

- `fraud_probability`: a score between `0` and `1` that represents the estimated fraud risk
- `decision`: the business action selected from the risk level
- `reason`: a short and meaningful justification for the decision
- `complaint_analysis`: additional insight from complaint text

The `reason` field should not be generic. It should explain why the decision was made. For example, it may refer to:

- the fraud probability exceeding a decision threshold
- suspicious transaction patterns
- inconsistency with expected transaction behavior
- complaint sentiment indicating urgency or concern
- supporting evidence from both transaction risk and complaint context

Example:

`Blocked because the fraud probability exceeded the high-risk threshold and the complaint indicates an unauthorized transaction.`

This improves transparency and makes system decisions easier to understand and justify.

## 8. Business Impact

The `BLOCK / REVIEW / APPROVE` design is important because real-world fraud systems require action, not only prediction.

- `BLOCK` helps stop highly suspicious transactions before loss increases
- `REVIEW` sends uncertain cases for human investigation instead of making an unsafe automatic decision
- `APPROVE` allows low-risk transactions to continue with less customer friction

This approach provides several practical benefits:

- reduced financial loss
- faster fraud response
- better analyst prioritization
- improved customer trust
- more useful integration of transaction signals and complaint context

## 9. Technical Approach

### 9.1 Role of Machine Learning
The machine learning model learns patterns from historical transaction data and estimates the probability that a new transaction is fraudulent.

### 9.2 Role of Decision Logic
Decision logic converts the fraud score into operational actions. Instead of producing only a binary label, the system uses risk thresholds to support `BLOCK`, `REVIEW`, and `APPROVE` decisions.

### 9.3 Role of NLP
The NLP module processes complaint text to identify sentiment and produce a concise summary. This gives extra context that may support fraud investigation and user-service understanding.

### 9.4 Role of API Deployment
FastAPI is used to expose the system as an application service. VPS deployment demonstrates how the solution can be accessed in a production-like environment rather than existing only as an offline notebook or script.

## 10. Academic Relevance

This project is academically relevant because it combines multiple areas of AI in one complete solution.

### 10.1 Tools and Techniques
The project covers important data science tasks such as:

- exploratory data analysis (EDA)
- data preprocessing
- machine learning model development
- evaluation of classification performance
- API deployment and system integration

### 10.2 NLP Relevance
The complaint analysis component maps directly to NLP topics such as:

- text processing
- sentiment analysis
- text summarization
- interpretation of unstructured data

### 10.3 University-Level Value
From an academic perspective, the project demonstrates:

- clear problem understanding
- integration of structured and unstructured data
- practical decision-system design
- end-to-end implementation from analysis to deployment

## 11. Assumptions and Limitations

### 11.1 Assumptions

- the transaction dataset is sufficiently representative of fraud behavior
- complaint text contains useful information for sentiment and summary generation
- decision thresholds can be defined meaningfully for business use
- the API receives data in the expected structure
- `V1` to `V28` are PCA-transformed features with confidential original names, which limits direct interpretability during EDA and model explanation

### 11.2 Limitations

- fraud datasets are usually highly imbalanced, which makes learning more difficult
- the available transaction dataset may not fully represent real banking operations
- anonymized features reduce interpretability
- complaint data in a project environment may be limited compared with real customer support systems
- decision rules may simplify real-world operational policies
- VPS deployment demonstrates usability, but not full enterprise-scale infrastructure

## 12. Final Summary

This project addresses an important real-world problem by combining fraud detection, decision support, complaint analysis, and deployment into one integrated AI system. The solution is valuable because it produces not only a fraud score, but also an actionable decision, a supporting reason, and complaint-based context.

From both industry and academic perspectives, the project demonstrates how machine learning, NLP, and deployment can work together to solve a realistic financial problem in a clear, structured, and practical way.

## 13. Next Step

The next notebook should be `02_data_overview.ipynb`. It should examine the dataset structure before detailed analysis and modeling by covering dataset shape, column names, data types, class imbalance, missing values, duplicate records, summary statistics, and an initial review of `Time`, `Amount`, and `Class`. This step will provide the foundation for later univariate, bivariate, multivariate, and modeling notebooks.

In [3]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Project root not found. Open the notebook from inside the project.")

project_root = find_project_root(Path.cwd())
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.data_loader import load_raw_data

df = load_raw_data()

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nClass distribution:")
print(df["Class"].value_counts())
print("\nClass distribution (%):")
print((df["Class"].value_counts(normalize=True) * 100).round(4))

ModuleNotFoundError: No module named 'pandas'